In [1]:
%%capture
pip install awswrangler folium scikit-learn branca shapely

In [2]:
import awswrangler as wr
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm
from sklearn.cluster import DBSCAN
from shapely.geometry import LineString, MultiPoint, Point
import math
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', None)

S3_BUCKET = "s3://aw-sbx-s3-sdl-riesgos-data-615299763803-01/discovery/anl_datstratapy"
PERIODO = 202512

In [3]:
%%capture
!pip install geopandas

### 01_predicciones_accionable_202512.csv

Contiene las predicciones del  universo sunat para las empresas al 202512, con esto se hace un inner join para obtener las predicciones de ventas para las empresas del universo accionable al periodo 202512

In [4]:
import awswrangler as wr
import pandas as pd

PERIODO = "202512"
DATABASE = "disc_datstratapy"
WORKGROUP = "ibk-discovery-datstratapy-workgroup"
RUTA_CSV = f"./01_predicciones_accionable_{PERIODO}.csv"

SQL = f"""
WITH universo AS (
    SELECT *
    FROM disc_datstratapy.t_bpe_bn_universo_accionable
    WHERE periodo = '{PERIODO}'
),
predicciones AS (
    SELECT *
    FROM disc_datstratapy.predicciones_universo
    WHERE codmes = '{PERIODO}'
)
SELECT
    u.*,
    p.codmes,
    p.key_value,
    p.filtro_deuda,
    p.y_pred_anual
FROM universo u
INNER JOIN predicciones p
    ON u.key_value_doc = p.key_value
"""

df = wr.athena.read_sql_query(
    sql=SQL,
    database=DATABASE,
    workgroup=WORKGROUP,
    ctas_approach=False
)

LIMITES_PRED = [0, 5000, 8333, 16667, 25000, 35000, 45000, 60000, 100000]
FACTORES_PRED = [2.11, 2.11, 1.58, 1.56, 1.53, 1.58, 1.48, 1.45, 1.10]

LIMITES_CLASICO = [0, 5000, 7500, 10000, 15000, 20000, 25000, 30000, 40000, 50000, 60000, 80000, 100000]
FACTORES_CLASICO = [6.56, 6.56, 5.05, 4.08, 3.09, 2.53, 2.23, 2.01, 1.77, 1.52, 1.51, 1.38, 1.10]

def obtener_factor(valor_mensual, filtro_deuda):
    if pd.isna(valor_mensual):
        return None

    es_no_bank = str(filtro_deuda).strip() == "No Bank"

    if es_no_bank:
        limites = LIMITES_CLASICO
        factores = FACTORES_CLASICO
    else:
        limites = LIMITES_PRED
        factores = FACTORES_PRED

    for i in range(len(limites) - 1):
        if limites[i] <= valor_mensual < limites[i + 1]:
            return factores[i]

    return factores[-1]

df["y_pred_mensual"] = df["y_pred_anual"] / 12

df["factor_informalidad"] = df.apply(
    lambda x: obtener_factor(x["y_pred_mensual"], x["filtro_deuda"]),
    axis=1
)

df["prediccion_con_factor_mensual"] = df["y_pred_mensual"] * df["factor_informalidad"]
df["prediccion_con_factor_anual"] = df["prediccion_con_factor_mensual"] * 12

df.to_csv(RUTA_CSV, index=False)

print(f"Registros: {len(df):,}")
print(f"Archivo guardado en: {RUTA_CSV}")

2026-04-17 03:17:45,639	WARNING services.py:2070 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 8235491328 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-04-17 03:17:45,777	INFO worker.py:1852 -- Started a local Ray instance.


Registros: 1,158,595
Archivo guardado en: ./01_predicciones_accionable_202512.csv


### _02_predicciones_accionable_coords_nonull.gpkg

Toma un unico registro por key value doc del universo accionable, solo toma las empresas que tienen coordenadas validas, y si la empresa tiene diferentes coordenadas en diferentes periodo se queda con la mas reciente, luego trae otras variables de la tabla de sus predicciones de ventas de cada empresa

In [5]:
import awswrangler as wr
import pandas as pd
import geopandas as gpd

PERIODO = "202512"
DATABASE = "disc_datstratapy"
WORKGROUP = "ibk-discovery-datstratapy-workgroup"
RUTA_GPKG = f"./_02_predicciones_accionable_coords_nonull.gpkg"

NOMBRE_CAPA = f"pred_{PERIODO}"

SQL_TABLA_ENRIQUECIDA = f"""
WITH universo AS (
    SELECT
        key_value_doc,
        ubigeo_departamento,
        ubigeo_provincia,
        ubigeo_distrito
    FROM (
        SELECT
            key_value_doc,
            ubigeo_departamento,
            ubigeo_provincia,
            ubigeo_distrito,
            ROW_NUMBER() OVER (
                PARTITION BY key_value_doc
                ORDER BY key_value_doc
            ) AS rn
        FROM disc_datstratapy.t_bpe_bn_universo_accionable
        WHERE periodo = '{PERIODO}'
    )
    WHERE rn = 1
),
coords AS (
    SELECT
        key_value,
        longitud_x,
        latitud_y
    FROM (
        SELECT
            key_value,
            longitud_x,
            latitud_y,
            ROW_NUMBER() OVER (
                PARTITION BY key_value
                ORDER BY codmes DESC
            ) AS rn
        FROM disc_datstratapy.t_bpe_lat_long_inf_ventas
        WHERE longitud_x IS NOT NULL
          AND latitud_y IS NOT NULL
    )
    WHERE rn = 1
),
predicciones AS (
    SELECT
        key_value,
        y_pred_anual,
        prm_ventas_u12m,
        filtro_deuda,
        flg_izi_pas,
        flg_bank,
        CASE
            WHEN sector_ciiu = 'pesca' THEN 'Pesca'
            WHEN sector_ciiu = 'transporte' THEN 'Transporte'
            WHEN sector_ciiu = 'manufactura' THEN 'Manufactura'
            WHEN sector_ciiu = 'otros_servicios' THEN 'Otros servicios'
            WHEN sector_ciiu = 'mineria_e_hidrocarburos' THEN 'Minería e hidrocarburos'
            WHEN sector_ciiu = 'electricidad_gas_y_agua' THEN 'Electricidad, gas y agua'
            WHEN sector_ciiu = 'alojamiento_y_restaurantes' THEN 'Alojamiento y restaurantes'
            WHEN sector_ciiu = 'construccion' THEN 'Construcción'
            WHEN sector_ciiu = 'telecomunicaciones' THEN 'Telecomunicaciones'
            WHEN sector_ciiu = 'comercio' THEN 'Comercio'
            WHEN sector_ciiu = 'administracion_publica_y_defensa' THEN 'Administración pública y defensa'
            WHEN sector_ciiu = 'agricultura' THEN 'Agricultura'
            WHEN sector_ciiu IS NULL OR TRIM(sector_ciiu) = '' THEN 'Sin información'
            ELSE 'Otros / revisar'
        END AS sector_ciiu,
        max_deuda_dct_u12m,
        total_pasivo_12um,
        sunat__persona_juridica,
        cash
    FROM disc_datstratapy.predicciones_universo
    WHERE codmes = '{PERIODO}'
)
SELECT
    p.key_value,
    u.ubigeo_departamento AS departamento,
    u.ubigeo_provincia AS provincia,
    u.ubigeo_distrito AS distrito,
    c.latitud_y,
    c.longitud_x,
    p.sector_ciiu,
    p.filtro_deuda,
    p.flg_izi_pas,
    p.flg_bank,
    p.sunat__persona_juridica,
    p.y_pred_anual,
    p.prm_ventas_u12m,
    p.max_deuda_dct_u12m,
    p.total_pasivo_12um,
    p.cash
FROM universo u
INNER JOIN predicciones p
    ON u.key_value_doc = p.key_value
INNER JOIN coords c
    ON u.key_value_doc = c.key_value
"""

df = wr.athena.read_sql_query(
    sql=SQL_TABLA_ENRIQUECIDA,
    database=DATABASE,
    workgroup=WORKGROUP,
    ctas_approach=False
)

LIMITES_PRED = [0, 5000, 8333, 16667, 25000, 35000, 45000, 60000, 100000]
FACTORES_PRED = [2.11, 2.11, 1.58, 1.56, 1.53, 1.58, 1.48, 1.45, 1.10]

LIMITES_CLASICO = [0, 5000, 7500, 10000, 15000, 20000, 25000, 30000, 40000, 50000, 60000, 80000, 100000]
FACTORES_CLASICO = [6.56, 6.56, 5.05, 4.08, 3.09, 2.53, 2.23, 2.01, 1.77, 1.52, 1.51, 1.38, 1.10]

def obtener_factor(valor_mensual, filtro_deuda):
    if pd.isna(valor_mensual):
        return None
    if str(filtro_deuda).strip() == "No Bank":
        limites = LIMITES_CLASICO
        factores = FACTORES_CLASICO
    else:
        limites = LIMITES_PRED
        factores = FACTORES_PRED
    for i in range(len(limites) - 1):
        if limites[i] <= valor_mensual < limites[i + 1]:
            return factores[i]
    return factores[-1]

df["y_pred_mensual"] = df["y_pred_anual"] / 12
df["factor_informalidad"] = df.apply(lambda x: obtener_factor(x["y_pred_mensual"], x["filtro_deuda"]), axis=1)
df["prediccion_con_factor_mensual"] = df["y_pred_mensual"] * df["factor_informalidad"]
df["prediccion_con_factor_anual"] = df["prediccion_con_factor_mensual"] * 12

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitud_x"], df["latitud_y"]),
    crs="EPSG:4326"
)

gdf = gdf.to_crs("EPSG:32718")

gdf.to_file(RUTA_GPKG, layer=NOMBRE_CAPA, driver="GPKG")

print(f"Coincidencias finales: {len(gdf):,}")
print(f"Negocios unicos: {gdf['key_value'].nunique():,}")
print(f"Archivo guardado en: {RUTA_GPKG}")

Coincidencias finales: 834,394
Negocios unicos: 834,394
Archivo guardado en: ./_02_predicciones_accionable_coords_nonull.gpkg


In [6]:
gdf.head()

,key_value,departamento,provincia,distrito,latitud_y,longitud_x,sector_ciiu,filtro_deuda,flg_izi_pas,flg_bank,...,y_pred_anual,prm_ventas_u12m,max_deuda_dct_u12m,total_pasivo_12um,cash,y_pred_mensual,factor_informalidad,prediccion_con_factor_mensual,prediccion_con_factor_anual,geometry
0,B41F24EF9BB30BE6A3BCCD5462E5E5A23E84A32CB3320673378B67CD4E97DB75,LIMA,CAETE,SAN VICENTE DE CAETE,-13.07543290,-76.39091791,Otros servicios,No Bank,Otros,02. No Bank Neg,...,"147,901.33",NaN,NaN,NaN,NaN,"12,325.11",4.08,"50,286.45","603,437.40",POINT (349199.266 8554108.249)
1,773F10FBF5DC94F8AF391C1554969628748BCE657E8DE7C0B962B5DDF46C9513,LIMA,LIMA,LOS OLIVOS,-11.97573327,-77.06057022,Manufactura,[Deuda 50K-2MM],Otros,01. Bank Neg.,...,"1,011,555.31",NaN,"390,047.03",NaN,NaN,"84,296.27",1.45,"122,229.60","1,466,755.16",POINT (275623.652 8675292.271)
2,DFF0E3EC8A7F6964C63D372E834D32381C600A807ED5D71657950D8F077BEE75,LIMA,LIMA,BREA,-12.05293061,-77.04391817,Manufactura,No Bank,Otros,01. Bank Neg.,...,"214,054.91",NaN,NaN,NaN,NaN,"17,837.91",3.09,"55,119.14","661,429.64",POINT (277501.063 8666764.423)
3,B32AB354E527177C85EE1A3929DD3C514DB347A55F232E7052A9BAE9ABF0B045,LA LIBERTAD,TRUJILLO,TRUJILLO,-8.11757298,-79.03068163,Minería e hidrocarburos,No Bank,<Pasivos Menor a 400K,02. No Bank Neg,...,"2,667,897.25",NaN,NaN,"68,985.74","366,897.23","222,324.77",1.10,"244,557.24","2,934,686.91",POINT (55595.908 9100494.236)
4,6320D48EE02E2DBEC374A9663787DD11B5827FB4C28E9E64BF40130729B358AB,LIMA,LIMA,SANTIAGO DE SURCO,-12.12112000,-77.03056000,Otros servicios,[Deuda 50K-2MM],<Pasivos Menor a 400K,01. Bank Neg.,...,"409,182.84",NaN,"297,314.15","39,239.90","12,248.00","34,098.57",1.53,"52,170.81","626,049.75",POINT (279011.795 8659230.581)
